In [137]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import numpy as np
from pygraphon.graphons import StepGraphon
from pygraphon.estimators import SmoothNetHist

from utils_generators import AssociativeFullyRandom
from pygraphon.norm import MseProbaEdge, MaeProbaEdge, AUCEdge, AUPRCEdge
import matplotlib.pyplot as plt
from pygraphon.plots.plot_matrices import spy


# Initialisation

In [ ]:
n = 100
k = 2
theta_intra_min, theta_intra_max = 0.5, 0.7
theta_inter_min, theta_inter_max = 0.05, 0.2
theta = AssociativeFullyRandom(
                    K=k,
                    min_inside_classes=theta_intra_min,
                    max_inside_classes=theta_intra_max,
                    min_outside_classes=theta_inter_min,
                    max_outside_classes=theta_inter_max,
                )
print(theta.Theta)
graphon = StepGraphon(theta.Theta, bandwidthHist=1 / k)
adjacency_matrix = graphon.draw(n=n, exchangeable=False)

# Olhede fit

In [ ]:
est = SmoothNetHist()
first_hist_olhede = est._first_approximate_graphon_from_adjacency(adjacencyMatrix= adjacency_matrix)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im2 = ax[2].imshow(first_hist_olhede[0].get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
_ = spy(adjacency_matrix, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("True graphon")
ax[2].set_title("Olhede histogram")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)
plt.show()

In [ ]:
from pygraphon.utils import edge_density
spy(adjacency_matrix)
print("Edge density: ", edge_density(adjacency_matrix[int(n/2):, int(n/2):]))
print("Edge density: ", edge_density(adjacency_matrix[:int(n/2), :int(n/2)]))
print("Edge density: ", edge_density(adjacency_matrix[:int(n/2), int(n/2):]))

# Smooth NetHist fit 

In [ ]:
my_est = SmoothNetHist()
my_est.fit(adjacency_matrix)

In [ ]:
my_est._first_fit_graphon

In [ ]:
print("Ratio of parameters:", my_est.get_ratio_par())
print("Number of unique parameters in SmoothNetHist:", len(np.unique(my_est.graphon.get_graphon())))
print("Number of unique parameters in Olhede:", len(np.unique(first_hist_olhede[0].get_graphon())))


In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im2 = ax[2].imshow(my_est._first_fit_graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im3 = ax[3].imshow(my_est.graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
_ = spy(adjacency_matrix, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("True graphon")
ax[2].set_title("Olhede histogram")
ax[3].set_title("SmoothNetHist")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)
plt.colorbar(im3, cax=cbar_ax)
plt.show()

In [ ]:
my_est._first_fit_graphon

In [ ]:
norm_olhede_mse = MseProbaEdge(n_nodes=n)(graphon, my_est._first_fit_graphon)
norm_olhede_mae = MaeProbaEdge(n_nodes=n)(graphon, my_est._first_fit_graphon)
norm_olhede_auc = AUCEdge(n_nodes=n)(adjacency_matrix, my_est._first_fit_graphon.get_edge_probabilities(n,False,True))
norm_olhede_auprc = AUPRCEdge(n_nodes=n)(adjacency_matrix, my_est._first_fit_graphon.get_edge_probabilities(n,False,True))

norm_smoothnet_mse = MseProbaEdge(n_nodes=n)(graphon, my_est.graphon)
norm_smoothnet_mae = MaeProbaEdge(n_nodes=n)(graphon, my_est.graphon)
norm_smoothnet_auc = AUCEdge(n_nodes=n)(adjacency_matrix, my_est.graphon.get_edge_probabilities(n,False,True))
norm_smoothnet_auprc = AUPRCEdge(n_nodes=n)(adjacency_matrix, my_est.graphon.get_edge_probabilities(n,False,True))

print("MSE Olhede vs SmoothNetHist:", norm_olhede_mse, norm_smoothnet_mse)
print("MAE Olhede vs SmoothNetHist:", norm_olhede_mae, norm_smoothnet_mae)
print("AUC Olhede vs SmoothNetHist:", norm_olhede_auc, norm_smoothnet_auc)
print("AUPRC Olhede vs SmoothNetHist:", norm_olhede_auprc, norm_smoothnet_auprc)

# SmoothNetHist step by step

In [ ]:
new_est = SmoothNetHist()
finite_est = new_est._approximate_graphon_from_adjacency(adjacencyMatrix= adjacency_matrix, number_link_communities= int((k+1)*k/2))


In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im2 = ax[2].imshow(my_est._first_fit_graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im3 = ax[3].imshow(my_est.graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im4 = ax[4].imshow(finite_est[0].get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
_ = spy(adjacency_matrix, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("True graphon")
ax[2].set_title("Olhede histogram")
ax[3].set_title("SmoothNetHist")
ax[4].set_title("SmoothNetHist finite")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)
plt.colorbar(im3, cax=cbar_ax)
plt.colorbar(im4, cax=cbar_ax)
plt.show()

In [ ]:
norm_smoothnet_mse_true = MseProbaEdge(n_nodes=n)(graphon, finite_est[0])
norm_smoothnet_mae_true = MaeProbaEdge(n_nodes=n)(graphon, finite_est[0])
norm_smoothnet_auc_true = AUCEdge(n_nodes=n)(adjacency_matrix, finite_est[0].get_edge_probabilities(n,False,True))
norm_smoothnet_auprc_true = AUPRCEdge(n_nodes=n)(adjacency_matrix, finite_est[0].get_edge_probabilities(n,False,True))

print("MSE Olhede vs SmoothNetHist vs Corrected:", norm_olhede_mse, norm_smoothnet_mse, norm_smoothnet_mse_true)
print("MAE Olhede vs SmoothNetHist vs Corrected:", norm_olhede_mae, norm_smoothnet_mae, norm_smoothnet_mae_true)
print("AUC Olhede vs SmoothNetHist vs Corrected:", norm_olhede_auc, norm_smoothnet_auc, norm_smoothnet_auc_true)
print("AUPRC Olhede vs SmoothNetHist vs Corrected:", norm_olhede_auprc, norm_smoothnet_auprc, norm_smoothnet_auprc_true)

# Other plots for paper